In [1]:
pip install bs4 requests

In [11]:
import json
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1. Daftar ID Sub-Genre Publik Netflix untuk memperbanyak pencarian URL
GENRE_IDS = [
    "83",     # TV Shows / Series
    "34399",  # Movies
    "1365",   # Action & Adventure
    "5763",   # Dramas
    "8711",   # Horror
    "6548",   # Comedies
    "8883",   # Romance
    "7424",   # Anime
    "6839",   # Documentaries
    "10118",  # Superhero
    "31574",  # Classics
    "5977",   # LGBTQ
    "783",    # Children & Family
    "8933",   # Thrillers
    "4370"    # Sports
]

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def collect_movie_urls(genre_list):
    """Mengumpulkan URL unik film/series dari banyak sub-genre Netflix."""
    movie_urls = set()
    print("=== STEP 1: Mengumpulkan URL dari Multi-Genre Netflix ===")

    for gid in genre_list:
        url = f"https://www.netflix.com/us/browse/genre/{gid}"
        print(f"Scraping daftar link dari Genre ID: {gid} ...")
        try:
            res = requests.get(url, headers=HEADERS, timeout=10)
            if res.status_code == 200:
                soup = BeautifulSoup(res.text, 'html.parser')
                for a in soup.find_all('a', href=True):
                    href = a['href']
                    if '/title/' in href:
                        clean_href = href.split('?')[0]
                        full_url = clean_href if clean_href.startswith('http') else f"https://www.netflix.com{clean_href}"
                        movie_urls.add(full_url)
            time.sleep(0.3) # Jeda waktu agar ramah terhadap server
        except Exception as e:
            print(f"Gagal mengambil dari Genre {gid}: {e}")

    print(f"\n[SELESAI STEP 1] Total URL film/series unik terkumpul: {len(movie_urls)}\n")
    return list(movie_urls)

def get_span_text(soup, header_text):
    """Helper aman untuk mengambil teks span setelah tag header tanpa AttributeError"""
    h_tag = soup.find(['h4', 'h3', 'span'], string=header_text)
    if h_tag and h_tag.find_next_sibling('span'):
        return h_tag.find_next_sibling('span').text.strip()
    return "NA"

def scrape_title_details(url):
    """Mengambil metadata detail tiap film/series dari halaman publik Netflix."""
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code != 200:
            return None, None

        soup = BeautifulSoup(res.text, 'html.parser')

        # Parse Microdata JSON-LD
        json_scripts = soup.find_all('script', type='application/ld+json')
        data = {}
        for script in json_scripts:
            if script.text and 'schema.org' in script.text:
                try:
                    data = json.loads(script.text)
                    break
                except Exception:
                    continue

        # Ekstraksi aman menggunakan .get() (mencegah KeyError)
        genre = data.get('genre', 'NA')
        release_date = data.get('dateCreated', 'NA')
        number_of_seasons = data.get('numberOfSeasons', 'NA')
        rating = data.get('contentRating', 'NA')
        synopsis = data.get('description', 'NA')

        # Ekstraksi HTML tambahan
        audio = get_span_text(soup, 'Audio')
        subtitles = get_span_text(soup, 'Subtitles')
        moods = get_span_text(soup, 'This show is ...')
        cast = get_span_text(soup, 'Cast')
        related_genres = get_span_text(soup, 'Genres')

        # Judul Film/Series
        title_name = soup.h1.text.strip() if soup.h1 else data.get('name', 'Unknown Title')

        details = {
            "Title": title_name,
            "Genre": genre if genre else "NA",
            "Synopsis": synopsis if synopsis else "NA",
            "Related Genres": related_genres,
            "Audio": audio,
            "Subtitles": subtitles,
            "Release Date": release_date if release_date else "NA",
            "Rating": rating if rating else "NA",
            "Mood": moods,
            "Seasons": number_of_seasons if number_of_seasons else "NA",
            "Cast": cast,
            "URL": url
        }

        # Bersihkan value kosong / NA
        clean_details = {k: v for k, v in details.items() if v not in ["NA", None, "", "Unknown"]}
        return clean_details, title_name

    except Exception as e:
        return None, None

def main():
    # 1. Kumpulkan semua URL dari belasan genre
    all_urls = collect_movie_urls(GENRE_IDS)

    if not all_urls:
        print("Tidak ada URL yang berhasil terkumpul.")
        return

    print("=== STEP 2: Scraping Detail Setiap Film/Series ===")
    results_dict = {}
    results_list = []

    total = len(all_urls)

    # Catatan: Jika ingin uji coba cepat, batasi looping ini, misal: all_urls[:50]
    for index, url in enumerate(all_urls, 1):
        print(f"[{index}/{total}] Fetching: {url}")
        details, title = scrape_title_details(url)

        if details and title:
            results_dict[title] = details
            results_list.append(details)

    # 3. Simpan Hasil ke JSON dan CSV
    print("\n=== STEP 3: Menyimpan Hasil Scraping ===")

    # Save JSON
    with open("netflix.json", "w", encoding="utf-8") as f:
        json.dump(results_dict, f, indent=4, ensure_ascii=False)
    print("[SUKSES] Tersimpan ke file 'netflix.json'")

    # Save CSV
    df = pd.DataFrame(results_list)
    df.to_csv("netflix.csv", index=False, encoding="utf-8")
    print("[SUKSES] Tersimpan ke file 'netflix.csv'")
    print(f"\n[SELESAI] Total {len(df)} data film/series berhasil diekstrak dengan sempurna!")

if __name__ == "__main__":
    main()

=== STEP 1: Mengumpulkan URL dari Multi-Genre Netflix ===
Scraping daftar link dari Genre ID: 83 ...
Scraping daftar link dari Genre ID: 34399 ...
Scraping daftar link dari Genre ID: 1365 ...
Scraping daftar link dari Genre ID: 5763 ...
Scraping daftar link dari Genre ID: 8711 ...
Scraping daftar link dari Genre ID: 6548 ...
Scraping daftar link dari Genre ID: 8883 ...
Scraping daftar link dari Genre ID: 7424 ...
Scraping daftar link dari Genre ID: 6839 ...
Scraping daftar link dari Genre ID: 10118 ...
Scraping daftar link dari Genre ID: 31574 ...
Scraping daftar link dari Genre ID: 5977 ...
Scraping daftar link dari Genre ID: 783 ...
Scraping daftar link dari Genre ID: 8933 ...
Scraping daftar link dari Genre ID: 4370 ...

[SELESAI STEP 1] Total URL film/series unik terkumpul: 2638

=== STEP 2: Scraping Detail Setiap Film/Series ===
[1/2638] Fetching: https://www.netflix.com/us/title/81406219
[2/2638] Fetching: https://www.netflix.com/us/title/80223371
[3/2638] Fetching: https://www.n

KeyboardInterrupt: 